# RAG experiments: memory LoRA vs retrieval vs both

Three systems answer the same 100 questions (`data/synthetic_question.json`):

| RAG | Generator | Retrieval |
|---|---|---|
| `memory_lora_only` | Qwen2.5-Coder-0.5B-Instruct + memory LoRA | none |
| `base_plus_retrieval` | Qwen2.5-Coder-0.5B-Instruct (no fine-tuning) | Chroma + ontology |
| `memory_lora_plus_retrieval` | Qwen2.5-Coder-0.5B-Instruct + memory LoRA | Chroma + ontology |

**Vector database (indexing):**
- The trained ontology extractor (Qwen2.5-0.5B-Instruct + `data/ontology_lora.bin`) tags every section with entities. `ontology.json` synonyms turn aliases into canonical names.
- Sections are embedded with `BAAI/bge-m3` in a LangChain `Chroma` collection, with the entities and provenance stored as metadata.

**Retrieval (query time):** ontology entities mentioned in the question boost sections that contain them. Their relationships from `ontology.json` are added to the prompt as known facts.

**Metrics:**
- cosine similarity (bge-m3) between each answer and the expected answer, unrounded
- seconds each RAG took to answer (retrieval + generation)

> Import `rag_etl` before `torch` (it sets torch environment variables needed on this machine).

In [1]:
from rag_etl import lora, rag  # first: sets torch env vars before torch is imported

import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.precision", 17)   # show metrics unrounded

QUESTIONS_PATH = "data/synthetic_question.json"
SECTIONS_PATH = "data/sections.parquet"
ONTOLOGY_PATH = "data/ontology.json"
GOLD_ENTITIES_PATH = "data/dataset_entities.parquet"
EXTRACTIONS_PATH = "data/sections_extracted_entities.parquet"   # cache of the extractor's output
CHROMA_DIR = "data/chroma"
RESULTS_PATH = "data/rag_eval_results.parquet"
REUSE_EXTRACTIONS = True   # set False to re-run the ontology extractor

K, FETCH_K, ENTITY_BOOST = 3, 10, 0.05

## 1. Questions
80 answerable questions written from the documentation, each with the section that answers it (`source`), plus 20 questions the documentation can't answer. Their expected answer is *"I don't have information about that topic"*.

In [2]:
questions = json.load(open(QUESTIONS_PATH))
qdf = pd.DataFrame(questions)
print(qdf["answerable"].value_counts().to_dict())
print(qdf.loc[qdf["answerable"], "source"].str.split("/").str[0].value_counts().to_dict())
qdf.sample(5, random_state=0)[["prompt", "answer", "answerable"]]

{True: 80, False: 20}
{'catalog-portfolio-api': 16, 'vendor-stockkeeper-api': 11, 'traffic-gate-api': 10, 'demand-forecast-docs': 10, 'payment-promise-gateway': 9, 'order-workflow-api': 9, '2p-revenue-optimizer-api': 6, 'zenith-keeper': 4, 'unified-transaction-configs': 3, 'price-engine-api': 2}


,prompt,answer,answerable
26,What file is needed to run order-workflow-api tests against gh-proxy?,A private key file at src/main/resources/workflow-gh-proxy.key.,True
86,What machine learning model replaced Prophet in the forecasting project in 2025?,I don't have information about that topic,False
2,Which endpoint of vendor-stockkeeper-api handles the Feed Consumer?,POST /fp/inventory/operations/feed.,True
55,What does the marketplace forecasting project forecast?,"The demand of products in the marketplace, giving TGMV and TSI for 9 product categories plus the...",True
75,What is the maximum size of a configuration JSON in unified-transaction-configs?,"50kb, a limitation of KVS.",True


## 2. Sections to index
Index `sections.parquet` rows with `For_training = 1` and `is_TO_DO = 0`: one copy of each near-duplicate group (the latest version), and no placeholders. Indexing every old copy would fill the top-k results with duplicates.

In [3]:
sections = pd.read_parquet(SECTIONS_PATH)
sections = sections[(sections["For_training"] == 1) & (sections["is_TO_DO"] == 0)].reset_index(drop=True)
print(f"{len(sections)} sections from {sections['path'].nunique()} files")

248 sections from 44 files


## 3. Ontology extraction with the trained adapter
Results are cached in `data/sections_extracted_entities.parquet` (about 4 minutes on the GPU). Truncated JSON outputs are recovered entity by entity.

In [4]:
onto = rag.OntologyIndex.from_file(ONTOLOGY_PATH)
keys = [rag.section_key(r) for _, r in sections.iterrows()]
cached = Path(EXTRACTIONS_PATH)
if REUSE_EXTRACTIONS and cached.exists() and set(pd.read_parquet(cached)["key"]) == set(keys):
    extractions = pd.read_parquet(cached)
    print("loaded cached extractions")
else:
    tok_onto = lora.load_tokenizer(lora.ONTOLOGY_MODEL)
    onto_model = lora.load_lora_weights(lora.ONTOLOGY_MODEL, "data/ontology_lora.bin")
    start = time.perf_counter()
    extractions = rag.extract_section_entities(sections, onto_model, tok_onto, batch_size=8)
    print(f"extracted {len(extractions)} sections in {time.perf_counter() - start:.0f}s")
    extractions.to_parquet(cached, index=False)
    del onto_model
    gc.collect(); torch.cuda.empty_cache()
print(f"parsed outputs: {extractions['parsed'].mean():.1%}")

loaded cached extractions
parsed outputs: 100.0%


In [5]:
# How close is the extractor to Claude's annotations? Entity-name precision / recall / F1,
# split by whether the section was in the adapter's training set.
gold = pd.read_parquet(GOLD_ENTITIES_PATH)
training_set, test_set, dev_set = lora.split_dataset(gold, seed=lora.SEED)
split_of = {**{rag.section_key(r): "train" for _, r in training_set.iterrows()},
            **{rag.section_key(r): "test" for _, r in test_set.iterrows()},
            **{rag.section_key(r): "dev" for _, r in dev_set.iterrows()}}
gold_by_key = {rag.section_key(r): r["entities"] for _, r in gold.iterrows()}
prf = pd.DataFrame(
    [(k, split_of[k], *rag.entity_prf(gold_by_key[k], e, onto)) for k, e in zip(extractions["key"], extractions["extracted"])],
    columns=["key", "split", "precision", "recall", "f1"],
)
prf.groupby("split")[["precision", "recall", "f1"]].mean()

,precision,recall,f1
split,,,
dev,0.59479365079365076,0.55544444444444441,0.54856265324558007
test,0.71180952380952378,0.69576190476190480,0.66280161668396953
train,0.81167603698504820,0.71979683033701469,0.74016171418915255


## 4. Vector database: bge-m3 embeddings in Chroma

In [6]:
embeddings = rag.load_embeddings()
docs = rag.build_documents(sections, extractions, onto)
store = rag.build_vector_store(docs, embeddings, CHROMA_DIR)
print(f"{store._collection.count()} documents in {CHROMA_DIR}")
print(docs[0].page_content[:600]); print(docs[0].metadata)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

248 documents in data/chroma
[2p-revenue-optimizer-api latest] README.md > Spring Boot App model for Java 17
We provide a basic model for JDK 17 / Spring based web applications.

Please address any questions and comments to [VendorFlow Issue Tracker](https://github.com/vendorflow/vendorflow/issues).
Entities: Java 17, Spring Boot, VendorFlow Issue Tracker
{'key': '2p-revenue-optimizer-api/latest/guide/README.md#Spring Boot App model for Java 17', 'path': '2p-revenue-optimizer-api/latest/guide/README.md', 'file_name': 'README.md', 'project_name': '2p-revenue-optimizer-api', 'version': 'latest', 'section': 'Spring Boot App model for Java 17', 'entities': 'Java 17 | Spring Boot | VendorFlow Issue Tracker'}


In [7]:
# Retrieval quality on the answerable questions: is the source section in the top K?
# Same retriever without the ontology (entity_boost = 0) as a baseline.
answerable = [q for q in questions if q["answerable"]]
def hit_rate(retriever):
    return np.mean([q["source"] in retriever.retrieve(q["prompt"]).keys for q in answerable])

retriever = rag.OntologyRetriever(store, onto, k=K, fetch_k=FETCH_K, entity_boost=ENTITY_BOOST)
plain = rag.OntologyRetriever(store, onto, k=K, fetch_k=FETCH_K, entity_boost=0.0)
pd.Series({f"hit@{K}, bge-m3 only": hit_rate(plain), f"hit@{K}, bge-m3 + ontology boost": hit_rate(retriever)})

hit@3, bge-m3 only                0.84999999999999998
hit@3, bge-m3 + ontology boost    0.82499999999999996
dtype: float64

In [8]:
example = retriever.retrieve(answerable[0]["prompt"])
print("Question:", answerable[0]["prompt"])
print("Ontology entities:", example.query_entities)
print(rag.format_context(example)[:1500])

Question: What is vendor-stockkeeper-api responsible for?
Ontology entities: ['vendor-stockkeeper-api']
Known facts:
- Error restock in vendor-stockkeeper-api monitors vendor-stockkeeper-api
- Stockkeeper-api - Error restocking item (negative quantity) monitors vendor-stockkeeper-api
- Validation stock (publication has more stock than warehouse's stock) monitors vendor-stockkeeper-api
- Validation stock ERROR updating item monitors vendor-stockkeeper-api
- vendor-stockkeeper-api deployed to production
- vendor-stockkeeper-api deployed to stage
- vendor-stockkeeper-api deployed to stream-prod
- vendor-stockkeeper-api exposes GET /fp/inventory/item/detail/{identifierRegister}

Document 1:
[vendor-stockkeeper-api latest] introduction.md > VENDOR STOCKKEEPER API > About
Stockkeeper-api is the application responsible for updating the stock of items belonging to the Standard area that are offered on TechZone. There are two main endpoints to perform restocking: one for 3PL stock movements (sm

## 5. The three RAGs
One Qwen2.5-Coder-0.5B-Instruct model is loaded with the memory adapter. `base_plus_retrieval` runs the same weights with the adapter switched off.

In [9]:
tok_coder = lora.load_tokenizer(lora.MEMORY_MODEL)
coder = lora.load_lora_weights(lora.MEMORY_MODEL, "data/memory_lora.bin")

rags = [
    rag.RAG("memory_lora_only", rag.Generator(coder, tok_coder, use_adapter=True)),
    rag.RAG("base_plus_retrieval", rag.Generator(coder, tok_coder, use_adapter=False), retriever),
    rag.RAG("memory_lora_plus_retrieval", rag.Generator(coder, tok_coder, use_adapter=True), retriever),
]
q = answerable[5]
print("Q:", q["prompt"], "\nExpected:", q["answer"])
for r in rags:
    out = r.run(q["prompt"])
    print(f"- {r.name} ({out['seconds']}s): {out['answer']}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Q: What are the database names of vendor-stockkeeper-api for test, stage and prod? 
Expected: inventtest for test, inventstg for stage and inventprd for prod, all on host proxysql.master.techzoneseginf.com port 6612.
- memory_lora_only (0.15960923099919455s): vendor-stockkeeper-api-test


- base_plus_retrieval (0.2870529159990838s): The database names of vendor-stockkeeper-api for test, stage, and prod are `inventtest`, `proxysql.master.techzoneseginf.com`, and `vendor-stockkeeper-stage.techzoneseginf.com`, respectively.
- memory_lora_plus_retrieval (0.16456967299745884s): Inventtest, inventstg, vendor-stockkeeper-api


## 6. Evaluation
For each question and RAG:
- `cosine_similarity`: bge-m3 cosine between the answer and the expected answer (not rounded)
- `seconds`: how long the RAG took to produce the answer (retrieval + generation)

In [10]:
results = rag.evaluate_rags(rags, questions, embeddings)
results.to_parquet(RESULTS_PATH, index=False)
results[["rag", "prompt", "answer", "cosine_similarity", "seconds"]].head(10)

memory_lora_only: 100 answers in 25.6s


base_plus_retrieval: 100 answers in 20.1s


memory_lora_plus_retrieval: 100 answers in 26.4s


,rag,prompt,answer,cosine_similarity,seconds
0,memory_lora_only,What is vendor-stockkeeper-api responsible for?,vendor-stockkeeper-api is a RESTful API service designed to manage and interact with the Vendor ...,0.52927753109507358,0.25197522499729530
1,memory_lora_only,Which endpoint of vendor-stockkeeper-api handles the Restock Consumer?,The endpoint of vendor-stockkeeper-api handles the Restock Consumer at `/vendor-stockkeeper/oper...,0.65542446747755667,0.25227972200082149
2,memory_lora_only,Which endpoint of vendor-stockkeeper-api handles the Feed Consumer?,The endpoint `/vendor-stockkeeper-api/operations/feed/consumer` is responsible for handling the ...,0.66887127589919049,0.25572302100044908
3,memory_lora_only,What is the production base URL of vendor-stockkeeper-api on the TechZone Platform?,The production base URL of vendor-stockkeeper-api on the TechZone Platform is `vendor-stockkeepe...,0.78693699530269723,0.27679455599718494
4,memory_lora_only,What field does the restock-new request body carry inside msg?,msg is a brand object,0.43390929676426820,0.06993104499997571
5,memory_lora_only,"What are the database names of vendor-stockkeeper-api for test, stage and prod?",vendor-stockkeeper-api-test,0.49066561222725130,0.06924110399995698
6,memory_lora_only,Which two inventory flows require verifying test items before executing actions?,- `inventory_flow_with_test_items`,0.56561989021117232,0.10229904800144141
7,memory_lora_only,How do you identify that an item is a test item in the restock flow?,"To identify that an item is a test item in the RESTOCK flow, you need to check the item's status...",0.71599455259913347,0.36309998100114171
8,memory_lora_only,"In the Feed flow, which table is queried with the operation id?",The table `operation_log` is queried with the operation id,0.50017654407152945,0.14404316200307221
9,memory_lora_only,What should you do when the alert 'Stockkeeper-api - Error restocking item (negative quantity)' ...,Check logs for details,0.50242059056155108,0.05814693300271756


In [11]:
summary = results.groupby("rag", sort=False).agg(
    cosine_mean=("cosine_similarity", "mean"),
    cosine_median=("cosine_similarity", "median"),
    seconds_mean=("seconds", "mean"),
    seconds_median=("seconds", "median"),
    seconds_total=("seconds", "sum"),
)
summary

,cosine_mean,cosine_median,seconds_mean,seconds_median,seconds_total
rag,,,,,
memory_lora_only,0.58783555614761229,0.57858256809650166,0.25618235558002200,0.18802507449981931,25.61823555800219765
base_plus_retrieval,0.64285746197849392,0.70040982483744174,0.20081734199015044,0.15113803350141097,20.08173419901504531
memory_lora_plus_retrieval,0.68199981095504381,0.70730383274936393,0.26427337974975673,0.21468617299979087,26.42733797497567139


In [12]:
by_type = results.assign(question_type=np.where(results["answerable"], "answerable", "unanswerable")).pivot_table(
    index="rag", columns="question_type", values="cosine_similarity", aggfunc="mean", sort=False
)
by_type["refusal rate on unanswerable"] = results[~results["answerable"]].groupby("rag", sort=False)["refused"].mean()
by_type["false refusals on answerable"] = results[results["answerable"]].groupby("rag", sort=False)["refused"].mean()
by_type["source retrieved (answerable)"] = results[results["answerable"]].groupby("rag", sort=False)["source_retrieved"].mean()
by_type

question_type,answerable,unanswerable,refusal rate on unanswerable,false refusals on answerable,source retrieved (answerable)
rag,,,,,
memory_lora_only,0.60429372003146598,0.52200290061219745,0.25000000000000000,0.0125,NaN
base_plus_retrieval,0.71432837251064774,0.35697381984987880,0.00000000000000000,0.0000,0.82499999999999996
memory_lora_plus_retrieval,0.70119732469545937,0.60520975599338156,0.40000000000000002,0.0125,0.82499999999999996


In [13]:
# Memory-only RAG: questions whose source section was in the memory adapter's training set vs held out
mem = results[(results["rag"] == "memory_lora_only") & results["answerable"]].copy()
mem["source split"] = mem["source"].map(split_of)
mem.groupby("source split")["cosine_similarity"].agg(["count", "mean", "median"])

,count,mean,median
source split,,,
dev,6,0.57254802393030080,0.55251214691708417
test,11,0.57506698999102779,0.60439770917160374
train,63,0.61242019950847892,0.58801487983745360


In [14]:
# Per-question cosine similarity and seconds, side by side
wide = results.pivot_table(index=["question_id", "prompt"], columns="rag", values=["cosine_similarity", "seconds"], sort=False)
wide.head(20)

cosine_similarity  \
rag                                                                                                                  memory_lora_only   
question_id prompt                                                                                                                      
0           What is vendor-stockkeeper-api responsible for?                                                       0.52927753109507358   
1           Which endpoint of vendor-stockkeeper-api handles the Restock Consumer?                                0.65542446747755667   
2           Which endpoint of vendor-stockkeeper-api handles the Feed Consumer?                                   0.66887127589919049   
3           What is the production base URL of vendor-stockkeeper-api on the TechZone Platform?                   0.78693699530269723   
4           What field does the restock-new request body carry inside msg?                                        0.43390929676426820   
5           What are the database names of vendor-stockkeeper-api for test, stage and prod?                       0.49066561222725130   
6           Which two inventory flows require verifying test items before executing actions?                      0.56561989021117232   
7           How do you identify that an item is a test item in the restock flow?                                  0.71599455259913347   
8           In the Feed flow, which table is queried with the operation id?                                       0.50017654407152945   
9           What should you do when the alert 'Stockkeeper-api - Error restocking item (negative quantity)' f...  0.50242059056155108   
10          Where should a ticket be raised to block a publication and prevent out-of-stock sales?                0.43492669187160349   
11          What is payment-promise-gateway?                                                                      0.58416852590357549   
12          Which Java version and framework does payment-promise-gateway use?                                    0.60439770917160374   
13          Which dependency injection library does payment-promise-gateway use?                                  0.49208504594016172   
14          Which endpoint creates a payment promise in payment-promise-gateway?                                  0.76460971787105181   
15          Which endpoint confirms a late fee forgiveness proposal?                                              0.84979959116411996   
16          What does GET /payment_promise/form return?                                                           0.70970400498792430   
17          Which header identifies the user in GET /payment_promise/active?                                      0.73217147408028294   
18          What environment variable must be set to run payment-promise-gateway locally?                         0.56699673673363971   
19          Which design pattern does PromiseCommand illustrate in payment-promise-gateway?                       0.67625154878776761   

                                                                                                                                       \
rag                                                                                                               base_plus_retrieval   
question_id prompt                                                                                                                      
0           What is vendor-stockkeeper-api responsible for?                                                       0.78469929454392096   
1           Which endpoint of vendor-stockkeeper-api handles the Restock Consumer?                                0.71895575298727676   
2           Which endpoint of vendor-stockkeeper-api handles the Feed Consumer?                                   0.61005886661367037   
3           What is the production base URL of vendor-stockkeeper-api on the TechZone Platform?                   0.78701692823717928   
4    

### Takeaways (100 questions, RTX 3080 Ti)

- **Retrieval plus the memory adapter scores highest:** mean cosine 0.682, against 0.643 for base + retrieval and 0.588 for memory only. On answerable questions both retrieval RAGs are close (0.714 base, 0.701 with adapter). The difference comes from the unanswerable questions: the adapter + retrieval refuses 40% of them, while the base model with retrieval never refuses (0%) and always answers from whatever context it was given.
- **The memory adapter alone mostly makes things up.** It answers fluently but with invented endpoints and names. It scores barely higher on questions whose source section it trained on (0.612) than on held-out ones (0.575). Next-token training on raw text didn't turn into question answering in chat format; that would take QA-format supervised fine-tuning on top.
- **Retrieval works; the ontology boost doesn't help yet.** The source section is in the top 3 for 85% of questions with bge-m3 alone and 82.5% with the entity boost. Most questions name their service (e.g. `vendor-stockkeeper-api`), and that entity appears in every section of the project, so the boost doesn't separate sections and sometimes promotes the wrong one. The facts drawn from `ontology.json` for a service are generic (alerts, environments). Next step: ignore project-level entities when boosting, and pick facts by relevance to the question instead of by frequency.
- **Speed is similar across the three:** a median of 0.15 to 0.21 s per answer. Base + retrieval is fastest because the unmerged LoRA adds compute at every layer and retrieval itself costs only milliseconds. Merging the adapter into the weights would remove that overhead.
- **The ontology extractor generalizes partly:** entity F1 against my annotations is 0.74 on sections it trained on, 0.66 on test sections and 0.55 on dev sections. All 248 outputs were valid JSON.
- **Cosine similarity is a weak judge.** A correct answer with extra words scores about 0.7, and a confident wrong answer in the right vocabulary about 0.55–0.6. Pair it with a factual check (key-fact match or an LLM judge) before deciding between systems.